In [1]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
path = 'datasets/aog'

df_names = ['components.csv', 'fleet_info.csv','maintenance_events.csv', 'sensor_readings.csv']
dfs = {}
for name in df_names:
    dfs[name] = pd.read_csv(os.path.join(path, name))

components, fleet_info, maintenance_events, sensor_readings = dfs.values()

In [6]:
components.head()

,component_id,aircraft_id,component_type,installation_date,status
0,C001,AC014,actuator,2023-08-06,active
1,C002,AC012,bearing,2023-07-11,active
2,C003,AC006,actuator,2023-02-05,replaced
3,C004,AC007,hydraulic_pump,2023-08-04,replaced
4,C005,AC015,bearing,2023-06-07,active


In [15]:
components[components['component_id'] == 'C001'] # FILTER STATUS == 'active'

,component_id,aircraft_id,component_type,installation_date,status
0,C001,AC014,actuator,2023-08-06,active


In [ ]:
fleet_info.head() # CALCULATE AGE OF AIRCRAFTS BASED ON MANUFACTURE DATE AND CURRENT DATE

,aircraft_id,aircraft_model,manufacture_year,base_airport,total_flight_hours
0,AC001,E190,2009,LIM,52814
1,AC002,B737-800,2017,SCL,45980
2,AC003,B737-800,2015,SCL,46680
3,AC004,A320,2005,SCL,32501
4,AC005,A320,2010,SCL,54382


In [ ]:
sensor_readings.head() # COMPONENTS ARE UNIQUE FOR EACH AIRCRAFT, TEMP, VIBRATION AND PRESSURE SHOULD BE TREATED AS MIN, MAX, MEAN AND STD FOR EACH COMPONENT

,reading_id,component_id,flight_cycle,timestamp,temperature_celsius,vibration_mm_s,pressure_psi
0,1,C001,1,2024-01-01 08:14:00,59.756919,2.815316,1943.090329
1,2,C001,2,2024-01-01 16:48:00,58.976611,2.752847,1958.279957
2,3,C001,3,2024-01-02 00:35:00,60.483151,3.086081,2052.898299
3,4,C001,4,2024-01-02 08:49:00,61.875312,3.336544,2046.006625
4,5,C001,5,2024-01-02 16:37:00,55.742453,3.294884,2013.584800


In [ ]:
maintenance_events # UNSCHEDULED IS THE EVENT WE NEED TO PREDICT

,event_id,component_id,flight_cycle,maintenance_type,action,downtime_hours,aircraft_id,event_date
0,1,C017,21.0,scheduled,cleaning,2.0,AC001,2024-01-08 04:00
1,2,C008,97.0,Scheduled,inspection,1.0,AC007,2024-02-02 12:00
2,3,C008,11.0,scheduled,adjustment,5.0,AC007,2024-01-04 20:00
3,4,C014,100.0,Scheduled,lubrication,2.0,AC005,2024-02-03 12:00
4,5,C012,98.0,UNSCHEDULED,replacement,30.0,AC018,2024-02-02 20:00
...,...,...,...,...,...,...,...,...
194,195,C027,18.0,SCHEDULED,inspection,1.0,AC002,2024-01-07 04:00
195,196,C034,46.0,SCHEDULED,inspection,4.0,AC010,2024-01-16 12:00
196,197,C020,4.0,SCHEDULED,adjustment,1.0,AC017,2024-01-02 12:00
197,198,C040,7.0,Scheduled,cleaning,4.0,AC002,2024-01-03 12:00


In [18]:
maintenance_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_id          199 non-null    int64  
 1   component_id      199 non-null    str    
 2   flight_cycle      197 non-null    float64
 3   maintenance_type  199 non-null    str    
 4   action            199 non-null    str    
 5   downtime_hours    199 non-null    float64
 6   aircraft_id       199 non-null    str    
 7   event_date        199 non-null    str    
dtypes: float64(2), int64(1), str(5)
memory usage: 12.6 KB


In [19]:
for name, df in dfs.items():
    print(f"\n{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(df.dtypes.to_string())


components.csv: 50 rows x 5 columns
component_id         str
aircraft_id          str
component_type       str
installation_date    str
status               str

fleet_info.csv: 20 rows x 5 columns
aircraft_id             str
aircraft_model          str
manufacture_year      int64
base_airport            str
total_flight_hours    int64

maintenance_events.csv: 199 rows x 8 columns
event_id              int64
component_id            str
flight_cycle        float64
maintenance_type        str
action                  str
downtime_hours      float64
aircraft_id             str
event_date              str

sensor_readings.csv: 5,015 rows x 7 columns
reading_id               int64
component_id               str
flight_cycle             int64
timestamp                  str
temperature_celsius    float64
vibration_mm_s         float64
pressure_psi           float64


In [20]:
null_pct = pd.concat(
    {name: df.isna().mean().mul(100).round(3) for name, df in dfs.items()}, names=["table", "column"]
).rename("null_%").to_frame()
null_pct

null_%
table                  column                     
components.csv         component_id          0.000
                       aircraft_id           0.000
                       component_type        0.000
                       installation_date     0.000
                       status                0.000
fleet_info.csv         aircraft_id           0.000
                       aircraft_model        0.000
                       manufacture_year      0.000
                       base_airport          0.000
                       total_flight_hours    0.000
maintenance_events.csv event_id              0.000
                       component_id          0.000
                       flight_cycle          1.005
                       maintenance_type      0.000
                       action                0.000
                       downtime_hours        0.000
                       aircraft_id           0.000
                       event_date            0.000
sensor_readings.csv    reading_id            0.000
                       component_id          0.000
                       flight_cycle          0.000
                       timestamp             0.000
                       temperature_celsius   1.994
                       vibration_mm_s        1.994
                       pressure_psi          1.994